# CHUNITHM result-photo extraction: visual walkthrough

This Jupyter Notebook serves as a visual walkthrough of the extraction pipeline. Follow a photo through validation, screen localization, OCR, anchor-based field crops, and final parsing.

**Setup:** 
1. Install the project with `.[ocr,notebook]` for CPU or `.[ocr-gpu,notebook]` for GPU. 
See the README for the GPU index. 

2. Select that environment as the notebook kernel.

3. Change the image path and device below, then run the cells in sequence. 
Note: Initialization can download model files.

Images are processed locally and are not uploaded by this project.

In [ ]:
# imports
from pathlib import Path
from unittest.mock import patch

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from chunithm_score_extractor import ChunithmExtractor
from chunithm_score_extractor.errors import ExtractionError
from chunithm_score_extractor.image import decode_image
from chunithm_score_extractor.layout import _collect_candidates, rectify_screen
from chunithm_score_extractor.ocr import PaddleOcrEngine
import chunithm_score_extractor.extractor as pipeline

In [ ]:
# Run from the repository root, or set an absolute path to your own photo.
IMAGE_PATH = Path('examples/img.jpg')
DEVICE = "cpu"  # You can also change this to either 'cpu' or 'gpu'.
MODEL_SIZE = "small" # You can change this to "tiny", "small", or "medium" depending on the OCR model size you want to use.
THRESHOLD = pipeline.CONFIDENCE_THRESHOLD
print(f'Image: {IMAGE_PATH.resolve()} | device: {DEVICE} | confidence threshold: {THRESHOLD}')

In [ ]:
# This is a helper function to display BGR images in Matplotlib
def show_bgr(ax, image, title):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')

## 1. Validate and decode the image

The extractor checks the file format, byte count and decoded dimensions. Then, the extractor applies EXIF orientation, and converts the image to OpenCV's BGR format. 

Invalid images raise a typed error instead of being processed.

`show_bgr` is a helper function that renders the BGR image using Matplotlib by converting it to RGB format.

In [ ]:
# Read and decode the image
image_bytes = IMAGE_PATH.read_bytes()
image = decode_image(image_bytes)

# Print basic information about the image
print(f'File: {len(image_bytes):,} bytes; decoded: {image.shape[1]} x {image.shape[0]} pixels')

# Display the initial image
fig, ax = plt.subplots(figsize=(12, 7))
show_bgr(ax, image, 'Initial image')
plt.show()

## 2. Locate and rectify the display

The locator uses grayscale, Gaussian blur and Canny edges, plus several thresholded and morphologically closed masks. Then, it collects convex four-corner candidates with plausible area and aspect ratio, and selects the largest candidate to perform the perspective warp.

The final perspective warp produces a rectified image of the display, which is then used for subsequent OCR and field extraction steps.

In [ ]:
area = image.shape[0] * image.shape[1]
candidates = []

# Covert the image to grayscale and apply edge detection
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blurred, 40, 130)

# Find contours in the edge-detected image and collect quadrilateral candidates
contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
_collect_candidates(contours, area, candidates)

# Apply multiple threshold levels and morphological closing to collect more candidates
close_size = min(61, max(15, int(min(image.shape[:2]) * 0.03) | 1))
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (close_size, close_size))
example_mask = None
for level in (40, 60, 80, 100, 120, 140, 150, 160, 180, 200, 220):
    mask = cv2.threshold(gray, level, 255, cv2.THRESH_BINARY)[1]
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    if level == 120: # You can change another level to visualize a different threshold
        example_mask = mask
    # Find contours and collect candidates for the current threshold level
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    _collect_candidates(contours, area, candidates)
print(f'Accepted quadrilateral candidates: {len(candidates)}')

# Visualize the collected candidates on the original image
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(edges, cmap='gray')
axes[0].set_title('Canny edges')
axes[1].imshow(example_mask, cmap='gray')
axes[1].set_title('Example mask (level 120)')
show_bgr(axes[2], image, 'Largest accepted candidate')
if candidates:
    corners = max(candidates, key=lambda candidate: candidate[0])[1]
    outline = np.vstack([corners, corners[0]])
    axes[2].plot(outline[:, 0], outline[:, 1], color='lime', linewidth=2)
else:
    axes[2].set_title('No candidate: localization will fail')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

# Rectify the screen based on the largest accepted candidate
screen = rectify_screen(image)  # Raises UnsupportedLayoutError if no candidate exists.

# Display the rectified screen
fig, ax = plt.subplots(figsize=(14, 8))
show_bgr(ax, screen, f'Rectified screen ({screen.shape[1]} x {screen.shape[0]})')
plt.show()

## 3. Run the actual extractor and record its OCR passes

We first run a full-screen OCR pass to detect the anchors and fields. Each field can use a confident full-screen reading if at least 80% of its box is inside the field region; otherwise its enlarged crop is read separately. Thus there are one to three OCR passes. 

The wrapper records the actual inputs and results without changing the model or the extraction logic. A typed extraction failure is shown explicitly.

Note: First initialization can take time, particularly on CPU.

Afterwards, we use the results to identify layout anchors for the score, rank, and difficulty fields. These anchors help determine whether the detected text is in the expected areas and meets the confidence requirements.

### How the anchors are selected

Coordinates are fractions of the rectified screen, measured from its top-left corner. Anchor search uses the **center of each OCR box**. Field shortcuts instead use the proportion of each box inside the field region. The confidence threshold is normally 0.75.

- **Score and rank:** After stripping whitespace and uppercasing, the text must be exactly `SCORE` or `RANK` and meet the confidence threshold. The score center must be within x=0.10–0.42, y=0.12–0.38, and the rank center within x=0.40–0.75, y=0.12–0.38. Rank must start to the right of score, and their vertical centers must differ by at most 0.06 of screen height. Exactly one compatible pair is required.
- **Title:** There is no OCR label named `TITLE`. Instead, the extractor looks for a supported difficulty in x=0.65–1.00, y=0.025–0.15. The difficulty must be confident and unambiguous, with exactly one matching box. That difficulty box is the title anchor: the title crop starts just below it.

A green box meets the confidence threshold but is not necessarily an anchor or a correct reading. If you have existing notebook outputs, rerun the cells in order to refresh them after code changes.

In [ ]:
# Set up OCR recording and extraction pipeline
class RecordingOcr:
    def __init__(self, engine):
        self.engine = engine
        self.passes = []

    def read(self, image):
        readings = tuple(self.engine.read(image))
        self.passes.append((image.copy(), readings))
        return readings

# Initialize the OCR engine and the extractor with the recording wrapper
engine = RecordingOcr(PaddleOcrEngine(size=MODEL_SIZE, device=DEVICE))
extractor = ChunithmExtractor(ocr=engine, confidence_threshold=THRESHOLD)
regions = {}
field_regions = {}
original_read_region = pipeline._read_region
original_field_regions = pipeline._field_regions

def record_field_regions(score_anchor, rank_anchor, difficulty_anchor):
    score_region, title_region = original_field_regions(score_anchor, rank_anchor, difficulty_anchor)
    field_regions.update(score=score_region, title=title_region)
    return score_region, title_region

# This wraps the original _read_region method to record the regions and their OCR readings
# for display and visualization purposes later on in this notebook
def record_region(ocr, screen_image, region):
    readings = original_read_region(ocr, screen_image, region)
    name = next(name for name, bounds in field_regions.items() if bounds == region)
    regions[name] = (engine.passes[-1][0], readings)
    return readings

# Record full-screen OCR and any crop retries needed by the extractor
result = None
failure = None
try:
    with patch.object(pipeline, '_field_regions', side_effect=record_field_regions), \
         patch.object(pipeline, '_read_region', side_effect=record_region):
        result = extractor.extract(image_bytes)
except ExtractionError as exc:
    failure = exc
    print(f'Extraction rejected: {type(exc).__name__}: {exc}')
print(f'Completed OCR passes: {len(engine.passes)}; completed field crops: {len(regions)}')

# render result
if engine.passes:
    full_image, full_readings = engine.passes[0]
    fig, ax = plt.subplots(figsize=(16, 10))
    show_bgr(ax, full_image, 'Full-screen OCR boxes')

    # iterate over readings
    for number, line in enumerate(full_readings, 1):
        color = 'lime' if line.confidence >= THRESHOLD else 'red'
        ax.add_patch(Rectangle((line.left, line.top), line.right - line.left,
                               line.bottom - line.top, fill=False, edgecolor=color, linewidth=1.5))
        ax.text(line.left, line.top, str(number), color='yellow', fontsize=8,
                bbox={'facecolor': 'black', 'alpha': 0.7, 'pad': 1})
        print(f'{number:>2}. {line.confidence:.3f}  {line.text}')
    plt.show()
    print(f'Green: >={THRESHOLD}. Red: <{THRESHOLD}')
    try:
        score_anchor, rank_anchor = pipeline._layout_anchors(full_readings, THRESHOLD)
        print(f'Layout anchors: {score_anchor.text!r}, {rank_anchor.text!r}')
        difficulty_lines = pipeline._in_region(full_readings, (0.65, 0.025, 1.0, 0.15))
        difficulty = pipeline._extract_difficulty(difficulty_lines, THRESHOLD)
        print(f'Difficulty candidate: {difficulty.value}')
    except ExtractionError as exc:
        print(f'Anchor/difficulty check rejected: {type(exc).__name__}: {exc}')
else:
    print('No completed OCR pass to display.')

## 4. Anchor-relative score and title crops

<!-- The extractor derives score bounds from the `SCORE` and `RANK` labels, and title bounds from the difficulty label. It enlarges each crop 2x with cubic interpolation before OCR. Boxes below show the actual normalized regions used by this run; the crop images are the exact inputs received by OCR, and their listed readings use crop coordinates. If an earlier stage failed, missing crops are reported rather than invented. -->

The extractor defines regions from the anchors and first checks full-screen readings. If a field cannot be determined confidently, it enlarges that region by 2x using cubic interpolation and performs OCR on the crop. Each field falls back independently.

The score region starts 0.02 screen widths left of the `SCORE` box and 0.025 screen heights below it, extends to 0.005 widths left of `RANK`, and ends 0.19 heights below `SCORE`. 

The title region starts 0.02 widths left and 0.002 heights below the difficulty box, extends to x=0.993, and ends 0.05 heights below the box. Bounds are clamped to the screen. The dashed boxes below identify the actual anchors, while the solid boxes show the field regions.

Full-screen candidates need at least 80% of their bounding-box area inside the field region. Fallback crop readings use their center for region membership. Title readings must also start no more than 0.027 screen heights below the difficulty box; vertical fragments centered inside a confidently detected vertical `CABINET-TO-CABINET` banner to the right of the difficulty are excluded.

In [ ]:
# This is done in the previous section where OCR readings and field crops are recorded

# Visualize the recorded field regions and the corresponding OCR crops
if field_regions:
    fig, ax = plt.subplots(figsize=(15, 9))
    show_bgr(ax, screen, 'Actual field regions on rectified screen')
    score_anchor, rank_anchor = pipeline._layout_anchors(full_readings, THRESHOLD)
    difficulty_lines = pipeline._in_region(full_readings, (0.65, 0.025, 1.0, 0.15))
    difficulty = pipeline._extract_difficulty(difficulty_lines, THRESHOLD)
    difficulty_anchor, = [line for line in difficulty_lines
                          if line.confidence >= THRESHOLD
                          and pipeline._difficulty_in_text(line.text) == {difficulty}]
    for name, anchor, color in (('score anchor', score_anchor, "red"), ('rank anchor', rank_anchor, "orange"),
                                ('difficulty anchor', difficulty_anchor, "yellow")):
        ax.add_patch(Rectangle((anchor.left, anchor.top), anchor.right - anchor.left,
                               anchor.bottom - anchor.top, fill=False, edgecolor=color,
                               linestyle='--', linewidth=2, label=name))
        print(f'{name}: {anchor.text!r}, confidence={anchor.confidence:.3f}, '
              f'box=({anchor.left:.1f}, {anchor.top:.1f}, {(anchor.right):.1f}, {anchor.bottom:.1f})')
    for name, color in (('score', 'lime'), ('title', 'cyan')):
        region = field_regions[name]
        left, top, right, bottom = region
        ax.add_patch(Rectangle((left * screen.shape[1], top * screen.shape[0]),
                               (right - left) * screen.shape[1], (bottom - top) * screen.shape[0],
                               fill=False, edgecolor=color, linewidth=2, label=name))
        print(f'{name} region (left, top, right, bottom): {region}')
    ax.legend()
    plt.show()
    for name in ('score', 'title'):
        first_pass = pipeline._overlapping_region(full_readings, field_regions[name])
        print(f'\n{name.title()} full-screen overlap candidates:')
        for line in first_pass:
            print(f'  {line.text!r}: confidence={line.confidence:.3f}')
        if name in regions:
            crop, readings = regions[name]
            fig, ax = plt.subplots(figsize=(14, 3))
            show_bgr(ax, crop, f'{name.title()} crop (2x)')
            plt.show()
            for line in readings:
                print(f'{name}: confidence={line.confidence:.3f}, text={line.text!r}')
            if not readings:
                print(f'{name}: OCR returned no text')
        else:
            print(f'{name}: no crop OCR completed (first pass accepted or extraction stopped)')
    if 'title' in regions:
        title_region = field_regions['title']
        _, title_readings = regions['title']
        title_in_region = pipeline._in_region(title_readings, title_region)
        print('\nTitle crop readings mapped back to screen coordinates:')
        for line in title_readings:
            in_region = line in title_in_region
            too_low = line.top / screen.shape[0] > difficulty_anchor.bottom / screen.shape[0] + 0.027
            banner = pipeline._is_cabinet_banner_fragment(line, full_readings, difficulty_anchor, THRESHOLD)
            reasons = []
            if not in_region: reasons.append('center outside title region')
            if too_low: reasons.append('below title row')
            if banner: reasons.append('confirmed vertical cabinet banner')
            if line.confidence < THRESHOLD: reasons.append('below confidence threshold')
            print(f'  {line.text!r}: confidence={line.confidence:.3f}, '
                  f'box=({line.left:.1f}, {line.top:.1f}, {line.right:.1f}, {line.bottom:.1f}); '
                  f'{", ".join(reasons) if reasons else "candidate for title parsing"}')
else:
    print('No field regions were determined: check the layout, difficulty, and error above.')

## 5. Parse or reject

The actual extractor filters readings by confidence and location. 

For score, the extractor looks for a confident numeric value within 0–1,010,000 and compares it with the top `SCORE:` header (excluding `HIGH SCORE`). If the first pass is unclear or conflicting, the score crop is retried. A known conflict cannot be resolved using the header alone. For title, the first eligible row is chosen; low-confidence or conflicting readings trigger a crop retry and, if unresolved, rejection rather than falling through to the artist below.

Ambiguous or unreadable fields produce a typed failure rather than a partial result. 

Note that OCR titles can still contain mistakes, wrong characters, or misread words, so a fuzzy search in further processing (e.g. when matching against a database of known songs) is recommended.

In [ ]:
if result is not None:
    print(f'Score:      {result.score:,}')
    print(f'Song title: {result.song_name}')
    print(f'Difficulty: {result.difficulty.value}')
elif failure is not None:
    print(f'No result: {type(failure).__name__}: {failure}')
else:
    raise RuntimeError('Run the extraction cell before inspecting the result.')